# **(( MONEY FLOW ))**

# OBJECTIVE:


 **Reconstruct the bank's approximate monthly
cash flow by standardizing its full transaction history
to a monthly average — at the bank level.**


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# DATA EXTRACTION:

In [ ]:
CSV_files_url = "https://raw.githubusercontent.com/aliabedalkereem-byte/Data-Analytics-Portfolio/main/5_Berka-Database-Deep-Analysis/CSV_files/"

TRANS= pd.read_csv(CSV_files_url  + "TRANS.csv")

/tmp/ipykernel_1285/2360837206.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  TRANS= pd.read_csv(CSV_files_url  + "TRANS.csv")


In [ ]:
T=TRANS.copy()[['account_id','date','type','operation','amount','k_symbol']]
T['date']=pd.to_datetime(T['date'])
T[:]

,account_id,date,type,operation,amount,k_symbol
0,1,1995-03-24,PRIJEM,VKLAD,1000,NaN
1,1,1995-04-13,PRIJEM,PREVOD Z UCTU,3679,NaN
2,1,1995-05-13,PRIJEM,PREVOD Z UCTU,3679,NaN
3,1,1995-06-13,PRIJEM,PREVOD Z UCTU,3679,NaN
4,1,1995-07-13,PRIJEM,PREVOD Z UCTU,3679,NaN
...,...,...,...,...,...,...
1056315,10451,1998-08-31,PRIJEM,NaN,62,UROK
1056316,10451,1998-09-30,PRIJEM,NaN,49,UROK
1056317,10451,1998-10-31,PRIJEM,NaN,34,UROK
1056318,10451,1998-11-30,PRIJEM,NaN,26,UROK


# CALCULATION:

In [ ]:
## MONTHS_COUNT (P)
min(T['date'])
max(T['date'])
P= (T['date'].max().year - T['date'].min().year) * 12 + (T['date'].max().month - T['date'].min().month) + 1
P
## TOTAL_INFLOW (IN)
IN=T.loc[(T['type']=='PRIJEM'),'amount'].sum()
IN
## TOTAL_OUTFLOW (OUT)
OUT=T.loc[(T['type']== 'VYDAJ')|(T['type']== 'VYBER'),'amount'].sum()
OUT
## fees + loan repayments (F & L)
F=T.loc[(T['k_symbol']=='SLUZBY'),'amount'].sum()
F
L=T.loc[(T['k_symbol']=='UVER'),'amount'].sum()
L

## MONTHLY MONEY FLOW — BANK LEVEL ##

print("MONTHLY MONEY FLOW APPROXIMATION :")
print("=" * 45)
print(f"IN/M     = {IN/P:>15,.0f} Kč  ← total inflow")
print(f"OUT/M    = {OUT/P:>15,.0f} Kč  ← total outflow")
print("-" * 45)
print(f"FEES/M   = {(F)/P:>15,.0f} Kč  ← fees ")
print(f"LOAN/M   = {(L)/P:>15,.0f} Kč  ← loan repayments")
print("-" * 45)
print(f"Bank/M   = {(F+L)/P:>15,.0f} Kč  ← fees + loan repayments")
print(f"Cust/M   = {(OUT-F-L)/P:>15,.0f} Kč  ← customer outflows")
print("=" * 45)
print(f"NET/M    = {(IN-OUT)/P:>15,.0f} Kč  ← remaining balance")

MONTHLY MONEY FLOW APPROXIMATION :
IN/M     =      44,826,148 Kč  ← total inflow
OUT/M    =      42,088,605 Kč  ← total outflow
---------------------------------------------
FEES/M   =          38,112 Kč  ← fees 
LOAN/M   =         767,403 Kč  ← loan repayments
---------------------------------------------
Bank/M   =         805,515 Kč  ← fees + loan repayments
Cust/M   =      41,283,090 Kč  ← customer outflows
NET/M    =       2,737,543 Kč  ← remaining balance


#VISUALIZATION:

In [ ]:
IN_M   = IN/P / 1_000_000
OUT_M  = OUT/P / 1_000_000
BANK_M = (F+L)/P / 1_000_000
CUST_M = (OUT-F-L)/P / 1_000_000
NET_M  = (IN-OUT)/P / 1_000_000

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=25,
        thickness=35,
        label=["INFLOW","OUTFLOW","BANK","Clients","NET" ],
        color=["#ffffff","#aaaaaa","#4fc3f7","#aaaaaa", "#81c784"],
        line=dict(color="black", width=0.5)
    ),
    link=dict(
        source=[0, 0, 1, 1],
        target=[1, 4, 2, 3],
        value =[OUT_M, NET_M, BANK_M, CUST_M],
        color =["rgba(180,180,180,0.2)", "rgba(129,199,132,0.3)","rgba(79,195,247,0.3)", "rgba(180,180,180,0.2)" ],
        label=[ f"{OUT_M:.2f}M Kč", f"{NET_M:.2f}M Kč", f"{BANK_M:.2f}M Kč", f"{CUST_M:.2f}M Kč" ]
    )
)])


fig.update_layout(
    title=dict(
        text=" <MONTHLY MONEY FLOW — BANK LEVEL  (Kč Millions)",
        font=dict(size=14, color="#ffffff"), x=0
    ),
    paper_bgcolor="#333333",
    plot_bgcolor="#1a1a1a",
    font=dict(size=12, color="#ffffff"),
    height=420,
    margin=dict(t=50, l=20, r=20, b=20)
)

fig.show()